# MALTO — v9 TPU: ModernBERT-base + R-Drop + EMA (TPU v5e-8)

**Target: beat 0.96423 (1st place) → 0.97+**

| Component | Why |
|---|---|
| R-Drop (kl=0.3, sequential) | Dropout-invariant features — fixes DeepSeek memorisation |
| EMA (decay=0.9995) | Validation & inference on smoothed weights |
| LDAM+DRW cap=**20×** | DeepSeek needs ~1.9× weight; 5× cap was the regression root cause |
| BF16 native | TPU v5e-8 has native BF16 — no GradScaler needed, faster than FP16 |
| Conservative post-proc | Temperature [0.6,1.5] + threshold [0.85,1.20] — fixes 0.027 OOF→public gap |
| Global blend only | Per-class Nelder-Mead overfit proven on 2400 samples |

**Hardware:** Kaggle TPU v5e-8 (8 cores × 16GB HBM = 128GB total)  
**TPU note:** Uses single XLA device per process. All `xm.mark_step()` calls flush the lazy execution graph.  
**Runtime:** ~60-90 min (5× faster than T4 x2 for matrix ops)


In [ ]:
import os, warnings, gc, time, re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack as sparse_hstack
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── TPU / GPU detection ──────────────────────────────────────────────────────
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    USE_TPU = True
    DEVICE  = xm.xla_device()
    N_GPUS  = 1
    USE_AMP = False
    # float32 on TPU: BF16 AdamW stores momentum in BF16 → rounds small updates to 0
    # TPU v5e-8 has 16GB/core, ModernBERT-base float32 = ~600MB → plenty of headroom
    DTYPE   = torch.float32
    print(f'TPU device: {DEVICE}  |  dtype: {DTYPE}  (float32 for optimizer stability)')
except ImportError:
    USE_TPU = False
    DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    N_GPUS  = torch.cuda.device_count() if torch.cuda.is_available() else 1
    USE_AMP = False
    DTYPE   = torch.float32
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32       = False
        print(f'GPU: {torch.cuda.get_device_name(0)} x {N_GPUS}')

SEED = 42
NUM_LABELS = 6
LABEL_MAP  = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}
np.random.seed(SEED)
torch.manual_seed(SEED)

KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input', '.', '..'
]
DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p; break
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root; break
assert DATA_DIR is not None, 'train.csv not found'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train_df['TEXT'] = train_df['TEXT'].fillna('empty')
test_df['TEXT']  = test_df['TEXT'].fillna('empty')

y_all       = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test  = test_df['TEXT'].values

print(f'Device: {DEVICE} | USE_TPU: {USE_TPU}')
print(f'Train: {len(train_df)}, Test: {len(test_df)}')
t_start = time.time()


In [ ]:
# ============================================================
# CONFIG — TPU v5e-8 (float32, stable EMA decay)
# ============================================================
MODEL_NAME  = 'answerdotai/ModernBERT-base'
MAX_LEN     = 512

MICRO_BATCH = 16 if USE_TPU else 4   # float32 + 16GB/core → still fits
GRAD_ACCUM  = 2  if USE_TPU else 4   # effective = 32

EPOCHS      = 10
LR          = 3e-5
PATIENCE    = 2
N_FOLDS     = 5
NUM_WORKERS = 4
DRW_CAP     = 20
TOP_K_CKPTS = 3
FULL_EPOCHS = 7
# EMA_DECAY fix: 0.9995 with only ~240 steps/fold → 88.7% init weights (broken)
# Rule: decay = 1 - 1/steps_per_fold ≈ 1 - 1/300 = 0.9967
# Use 0.99 (responsive for short training, half-life ~69 steps)
EMA_DECAY   = 0.99
KL_WEIGHT   = 0.3
ALPHA       = 0.6

EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM * N_GPUS
print(f'Model:      {MODEL_NAME}')
print(f'Batch:      {MICRO_BATCH}/device × {N_GPUS} × grad_accum {GRAD_ACCUM} = {EFFECTIVE_BATCH} effective')
print(f'LR: {LR}  |  Folds: {N_FOLDS}  |  DRW cap: {DRW_CAP}×  |  SWA top-{TOP_K_CKPTS}')
print(f'EMA_DECAY={EMA_DECAY} (half-life≈{int(0.693/(1-EMA_DECAY))} steps)  |  R-Drop kl={KL_WEIGHT}')
print(f'dtype: {DTYPE}  (float32 = stable AdamW momentum on TPU)')


In [ ]:
# ============================================================
# LDAM + DRW LOSS
# ============================================================
class LDAMLoss(nn.Module):
    def __init__(self, class_counts, max_margin=0.5,
                 drw_epoch=2, drw_ramp_epochs=3, label_smoothing=0.1):
        super().__init__()
        safe = np.maximum(class_counts, 1).astype(np.float64)
        margins = np.clip(max_margin / np.power(safe, 0.25), 0, 1.0)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.drw_epoch = drw_epoch; self.drw_ramp_epochs = drw_ramp_epochs
        self.current_epoch = 0; self.label_smoothing = label_smoothing
        inv = 1.0 / safe; w = inv / inv.sum() * len(class_counts)
        w   = np.clip(w, w.min(), w.min() * DRW_CAP)   # 20× cap for DeepSeek
        w   = w / w.sum() * len(class_counts)
        w   = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        self.register_buffer('drw_weights', torch.tensor(w, dtype=torch.float32))

    def set_epoch(self, e): self.current_epoch = e

    def _blended_weight(self, device):
        ep = self.current_epoch
        if ep < self.drw_epoch: return None
        t = min((ep - self.drw_epoch) / max(self.drw_ramp_epochs, 1), 1.0)
        return (torch.ones_like(self.drw_weights) + t * (self.drw_weights - 1)).to(device)

    def forward(self, logits, targets):
        logits = logits.float()
        if torch.isnan(logits).any(): logits = torch.nan_to_num(logits, nan=0.0)
        m = self.margins.to(logits.device)[targets]
        adj = logits.clone()
        adj[torch.arange(len(targets), device=logits.device), targets] -= m
        w    = self._blended_weight(logits.device)
        loss = F.cross_entropy(adj, targets, weight=w, label_smoothing=self.label_smoothing)
        if torch.isnan(loss): return F.cross_entropy(logits, targets)
        return loss


# ============================================================
# R-DROP KL DIVERGENCE
# Bidirectional KL between two stochastic forward passes.
# Forces dropout-invariant representations — critical for 80-sample DeepSeek.
# ============================================================
def rdrop_kl(logits1, logits2, kl_weight=0.3):
    p  = F.log_softmax(logits1.float(), dim=-1)
    q  = F.softmax(logits2.float(),     dim=-1)
    q_ = F.log_softmax(logits2.float(), dim=-1)
    p_ = F.softmax(logits1.float(),     dim=-1)
    kl = (F.kl_div(p, q, reduction='batchmean') +
          F.kl_div(q_, p_, reduction='batchmean')) * 0.5
    return kl_weight * kl


# ============================================================
# EMA UTILITIES
# dtype-safe load: respects original dtype per tensor.
# ============================================================
def _ema_load(model, ema_state):
    target_sd = model.state_dict()
    load_dict = {}
    for k, v in ema_state.items():
        orig = target_sd[k]
        load_dict[k] = v.to(dtype=orig.dtype, device=orig.device) \
                       if orig.is_floating_point() else orig
    model.load_state_dict(load_dict)


print(f'LDAMLoss ready | DRW cap={DRW_CAP}× | R-Drop kl={KL_WEIGHT} | EMA decay={EMA_DECAY}')


In [ ]:
# ============================================================
# DATASET + OPTIMIZER
# ============================================================
class TextDataset(Dataset):
    def __init__(self, enc, labels=None, indices=None):
        self.enc = enc; self.labels = labels; self.indices = indices
    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.enc['input_ids'])
    def __getitem__(self, idx):
        i = self.indices[idx] if self.indices is not None else idx
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item


def get_optimizer(model, lr, llrd=0.9):
    no_decay   = ['bias', 'LayerNorm', 'layernorm', 'layer_norm']
    params     = []
    head_params = [(n, p) for n, p in model.named_parameters()
                   if 'classifier' in n or 'head' in n]
    if head_params:
        params += [
            {'params': [p for n, p in head_params if not any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': 0.01},
            {'params': [p for n, p in head_params if     any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': 0.0},
        ]
    layer_nums = set()
    for n, _ in model.named_parameters():
        m = re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m: layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 22
    for i in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd ** (num_layers - 1 - i))
        lp = [(n, p) for n, p in model.named_parameters()
              if f'encoder.layer.{i}.' in n or f'layers.{i}.' in n]
        if lp:
            params += [
                {'params': [p for n, p in lp if not any(nd in n for nd in no_decay)],
                 'lr': layer_lr, 'weight_decay': 0.01},
                {'params': [p for n, p in lp if     any(nd in n for nd in no_decay)],
                 'lr': layer_lr, 'weight_decay': 0.0},
            ]
    emb_lr  = lr * (llrd ** num_layers)
    emb_p   = [(n, p) for n, p in model.named_parameters() if 'embed' in n.lower()]
    if emb_p:
        params += [
            {'params': [p for n, p in emb_p if not any(nd in n for nd in no_decay)],
             'lr': emb_lr, 'weight_decay': 0.01},
            {'params': [p for n, p in emb_p if     any(nd in n for nd in no_decay)],
             'lr': emb_lr, 'weight_decay': 0.0},
        ]
    assigned = set(id(p) for g in params for p in g['params'])
    rem = [p for n, p in model.named_parameters()
           if id(p) not in assigned and p.requires_grad]
    if rem: params.append({'params': rem, 'lr': lr * 0.5, 'weight_decay': 0.01})
    return torch.optim.AdamW(params)

print('Dataset + optimizer ready.')


In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================
print(f'Tokenizing with {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_enc = tokenizer(list(texts_train), max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
test_enc  = tokenizer(list(texts_test),  max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
print(f'Train: {train_enc["input_ids"].shape}, Test: {test_enc["input_ids"].shape}')


In [ ]:
# ============================================================
# TPU VERIFICATION  — confirms TPU is active
# ============================================================
if USE_TPU:
    # 1. Tensor device check
    t = torch.ones(4, 4).to(DEVICE)
    print(f'Tensor device : {t.device}')          # must be xla:0

    # 2. XLA available devices
    print(f'XLA devices   : {xm.get_xla_supported_devices()}')

    # 3. Timing probe — use .sum() to get a scalar, then .item() forces sync
    import time
    t2 = (t @ t).sum()                            # scalar, queued but not yet executed
    t0 = time.time(); _ = t2.item(); t1 = time.time()
    print(f'Sync call 1   : {(t1-t0)*1000:.1f}ms  (executes queued ops + syncs)')
    t0 = time.time(); _ = t2.item(); t1 = time.time()
    print(f'Sync call 2   : {(t1-t0)*1000:.1f}ms  (already synced, near 0ms)')

    print('\nTPU confirmed — E1 slow (XLA compilation), E2+ fast (3-4x speedup = XLA proof)')
    xm.mark_step()
else:
    print(f'GPU device: {torch.cuda.get_device_name(0)} x {N_GPUS}')


In [ ]:
# ============================================================
# K-FOLD TRAINING  —  R-Drop + EMA + SWA top-K  (TPU-safe)
# ============================================================
def _optimizer_step(optimizer):
    if USE_TPU:
        xm.optimizer_step(optimizer)
        xm.mark_step()
    else:
        optimizer.step()

def _empty_cache():
    if not USE_TPU and torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def load_model():
    kwargs = dict(num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    if USE_TPU:
        kwargs['attn_implementation'] = 'eager'   # no dynamic shapes
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **kwargs)
    if USE_TPU:
        model.config.unpad_inputs    = False   # no torch.nonzero → static graph
        model.config.reference_compile = False  # no torch.compile on XLA
        # model stays float32 — BF16 AdamW rounds small updates to 0
    else:
        base = model.model if hasattr(model, 'model') else model
        base.gradient_checkpointing_enable()
        if N_GPUS > 1:
            model = nn.DataParallel(model)
    return model


def train_fold(fold_idx, train_idx, val_idx):
    _empty_cache()
    print(f'\n{"="*55}')
    print(f'FOLD {fold_idx+1}/{N_FOLDS} | Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*55}')

    pin = not USE_TPU
    train_loader = DataLoader(
        TextDataset(train_enc, y_all, train_idx),
        batch_size=MICRO_BATCH, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
    val_loader  = DataLoader(
        TextDataset(train_enc, y_all, val_idx),
        batch_size=MICRO_BATCH * 4,
        num_workers=NUM_WORKERS, pin_memory=pin)
    test_loader = DataLoader(
        TextDataset(test_enc),
        batch_size=MICRO_BATCH * 4,
        num_workers=NUM_WORKERS, pin_memory=pin)

    model = load_model()
    model.to(DEVICE)

    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts); criterion.to(DEVICE)
    print(f'  DRW weights: {criterion.drw_weights.cpu().numpy().round(3)}')
    if USE_TPU:
        print(f'  attn: eager | unpad_inputs: False | dtype: float32 | EMA half-life: ~{int(0.693/(1-EMA_DECAY))} steps')

    actual_model = model.module if hasattr(model, 'module') else model
    optimizer    = get_optimizer(actual_model, lr=LR, llrd=0.9)
    n_steps      = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler    = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * n_steps), num_training_steps=n_steps)

    # EMA on CPU float32, decay=0.99 (half-life ~69 steps — correct for ~300 steps/fold)
    ema_state = {k: v.cpu().clone().float() if v.is_floating_point() else v.cpu().clone()
                 for k, v in actual_model.state_dict().items()}
    best_ema_state = None
    top_ckpts = []; best_f1 = 0.0; patience_ctr = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)
        drw_str = '  0%' if epoch < criterion.drw_epoch else \
            f'{int(min((epoch-criterion.drw_epoch)/criterion.drw_ramp_epochs,1)*100):3d}%'

        model.train()
        total_loss = 0.0; valid_steps = 0
        optimizer.zero_grad()

        loader_iter = pl.MpDeviceLoader(train_loader, DEVICE) if USE_TPU else train_loader

        for step, batch in enumerate(tqdm(loader_iter, desc=f'E{epoch+1}', leave=False)):
            inputs = batch if USE_TPU else {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')

            # ── R-Drop pass 1 ────────────────────────────────────────────────
            out1    = model(**inputs)
            logits1 = out1.logits.float()
            loss1   = criterion(logits1, labels) * 0.5 / GRAD_ACCUM
            l1_val  = loss1.item()
            logits1_det = logits1.detach()
            del out1, logits1
            if not (torch.isnan(loss1) or torch.isinf(loss1)):
                loss1.backward()
            del loss1
            if USE_TPU: xm.mark_step()

            # ── R-Drop pass 2 + KL ───────────────────────────────────────────
            out2    = model(**inputs)
            logits2 = out2.logits.float()
            loss2   = criterion(logits2, labels) * 0.5 / GRAD_ACCUM
            kl      = rdrop_kl(logits1_det, logits2, KL_WEIGHT) / GRAD_ACCUM
            loss_p2 = loss2 + kl
            l2_val  = loss_p2.item()
            del out2, logits2, logits1_det
            if not (torch.isnan(loss_p2) or torch.isinf(loss_p2)):
                loss_p2.backward()
            del loss_p2
            if USE_TPU: xm.mark_step()

            total_loss  += (l1_val + l2_val) * GRAD_ACCUM
            valid_steps += 1

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                _optimizer_step(optimizer)
                optimizer.zero_grad()
                scheduler.step()
                with torch.no_grad():
                    for k, v in actual_model.state_dict().items():
                        if ema_state[k].is_floating_point():
                            ema_state[k].mul_(EMA_DECAY).add_(
                                v.cpu().float(), alpha=1.0 - EMA_DECAY)

        # ── Validation on CURRENT model weights (not EMA) ──────────────────
        # Reason: EMA_DECAY=0.9995 with only ~300 steps/fold means EMA≈88% init.
        # We validate on live weights for honest F1 / early stopping.
        # EMA checkpoint is SAVED when live F1 improves (EMA will generalise better
        # at inference time once it has enough steps to converge).
        model.eval()
        preds, lbls = [], []
        val_iter = pl.MpDeviceLoader(val_loader, DEVICE) if USE_TPU else val_loader
        with torch.no_grad():
            for batch in val_iter:
                inp = batch if USE_TPU else {k: v.to(DEVICE) for k, v in batch.items()}
                lbl = inp.pop('labels')
                out = model(**inp)
                preds.extend(out.logits.float().argmax(-1).cpu().numpy())
                lbls.extend(lbl.cpu().numpy())
                if USE_TPU: xm.mark_step()

        val_f1 = f1_score(lbls, preds, average='macro')

        # Save EMA snapshot when live-model F1 is the best seen
        ema_ckpt = {k: v.clone() for k, v in ema_state.items()}
        if len(top_ckpts) < TOP_K_CKPTS or val_f1 > top_ckpts[-1][0]:
            if len(top_ckpts) >= TOP_K_CKPTS: top_ckpts.pop()
            top_ckpts.append((val_f1, ema_ckpt))
            top_ckpts.sort(key=lambda x: x[0], reverse=True)

        if val_f1 > best_f1:
            best_f1 = val_f1; patience_ctr = 0; marker = ' ** BEST **'
            best_ema_state = ema_ckpt
        else:
            patience_ctr += 1; marker = f' (p={patience_ctr}/{PATIENCE})'

        print(f'  E{epoch+1} | loss={total_loss/max(valid_steps,1):.4f} | val_f1={val_f1:.4f} | '
              f'DRW={drw_str} | {time.time()-t0:.0f}s{marker}')
        if patience_ctr >= PATIENCE: print(f'  Early stop at E{epoch+1}'); break

    # ── SWA: average top-K EMA checkpoints ───────────────────────────────────
    n_avg = len(top_ckpts)
    print(f'  SWA top-{n_avg} EMA: {[f"{f:.4f}" for f, _ in top_ckpts]}')
    avg_state = {k: (sum(c[k].float() for _, c in top_ckpts) / n_avg
                     ).to(top_ckpts[0][1][k].dtype) for k in top_ckpts[0][1]}
    _ema_load(actual_model, avg_state)
    if USE_TPU: xm.mark_step()
    model.eval()

    val_logits, test_logits = [], []
    with torch.no_grad():
        v_iter = pl.MpDeviceLoader(val_loader,  DEVICE) if USE_TPU else val_loader
        t_iter = pl.MpDeviceLoader(test_loader, DEVICE) if USE_TPU else test_loader
        for batch in v_iter:
            b = batch if USE_TPU else {k: v.to(DEVICE) for k, v in batch.items()}
            b.pop('labels', None)
            val_logits.extend(model(**b).logits.float().cpu().numpy())
            if USE_TPU: xm.mark_step()
        for batch in t_iter:
            b = batch if USE_TPU else {k: v.to(DEVICE) for k, v in batch.items()}
            test_logits.extend(model(**b).logits.float().cpu().numpy())
            if USE_TPU: xm.mark_step()

    del model, optimizer, scheduler, top_ckpts, avg_state, ema_state
    _empty_cache()
    return np.array(val_logits), np.array(test_logits), best_f1


# ── Run K-Fold ────────────────────────────────────────────────────────────────
skf             = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_logits      = np.zeros((len(y_all), NUM_LABELS))
test_logits_sum = np.zeros((len(texts_test), NUM_LABELS))
fold_scores     = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    vl, tl, bf = train_fold(fold_idx, train_idx, val_idx)
    oof_logits[val_idx] = vl
    test_logits_sum    += tl
    fold_scores.append(bf)
    print(f'  Fold {fold_idx+1} best F1: {bf:.4f} | Elapsed: {(time.time()-t_start)/60:.1f}min')

test_logits_fold = test_logits_sum / N_FOLDS
print(f'\n{N_FOLDS}-FOLD COMPLETE')
print(f'  Scores: {[f"{s:.4f}" for s in fold_scores]}')
print(f'  Mean:   {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'  Time:   {(time.time()-t_start)/60:.1f} min')


In [ ]:
# ============================================================
# FULL-DATA MODEL  (EMA + SWA last 3 epochs)  TPU-safe
# ============================================================
def train_full_data():
    _empty_cache()
    print(f'\n{"="*55}\nFULL-DATA | {len(y_all)} samples | {FULL_EPOCHS} epochs\n{"="*55}')

    pin     = not USE_TPU
    all_idx = np.arange(len(y_all))
    tr_load = DataLoader(TextDataset(train_enc, y_all, all_idx),
        batch_size=MICRO_BATCH, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
    te_load = DataLoader(TextDataset(test_enc),
        batch_size=MICRO_BATCH * 4,
        num_workers=NUM_WORKERS, pin_memory=pin)

    model        = load_model()
    model.to(DEVICE)
    fold_counts  = np.bincount(y_all, minlength=NUM_LABELS).astype(float)
    criterion    = LDAMLoss(fold_counts, drw_epoch=1); criterion.to(DEVICE)
    actual_model = model.module if hasattr(model, 'module') else model
    optimizer    = get_optimizer(actual_model, lr=LR * 0.8)
    n_steps      = (len(tr_load) // GRAD_ACCUM) * FULL_EPOCHS
    scheduler    = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * n_steps), num_training_steps=n_steps)
    ema_state = {k: v.cpu().clone().float() if v.is_floating_point() else v.cpu().clone()
                for k, v in actual_model.state_dict().items()}
    swa_states = []

    for epoch in range(FULL_EPOCHS):
        t0 = time.time(); criterion.set_epoch(epoch)
        model.train(); total_loss = 0.0; valid_steps = 0
        optimizer.zero_grad()

        tr_iter = pl.MpDeviceLoader(tr_load, DEVICE) if USE_TPU else tr_load
        for step, batch in enumerate(tqdm(tr_iter, desc=f'Full E{epoch+1}', leave=False)):
            inputs = batch if USE_TPU else {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            out  = model(**inputs)
            loss = criterion(out.logits.float(), labels) / GRAD_ACCUM
            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                total_loss += loss.item() * GRAD_ACCUM; valid_steps += 1
            del out
            if USE_TPU: xm.mark_step()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                _optimizer_step(optimizer)
                optimizer.zero_grad(); scheduler.step()
                with torch.no_grad():
                    for k, v in actual_model.state_dict().items():
                        if ema_state[k].is_floating_point():
                            ema_state[k].mul_(EMA_DECAY).add_(
                                v.cpu().float(), alpha=1.0 - EMA_DECAY)

        swa_tag = ''
        if epoch >= FULL_EPOCHS - 3:
            swa_states.append({k: v.clone() for k, v in ema_state.items()})
            swa_tag = ' [SWA+EMA]'
        print(f'  Full E{epoch+1}/{FULL_EPOCHS} | loss={total_loss/max(valid_steps,1):.4f} | '
              f'{time.time()-t0:.0f}s{swa_tag}')

    avg_state = {k: (sum(s[k].float() for s in swa_states) / len(swa_states)
                     ).to(swa_states[0][k].dtype) for k in swa_states[0]}
    _ema_load(actual_model, avg_state)
    if USE_TPU: xm.mark_step()
    model.eval()

    test_logits = []
    te_iter = pl.MpDeviceLoader(te_load, DEVICE) if USE_TPU else te_load
    with torch.no_grad():
        for batch in te_iter:
            b = batch if USE_TPU else {k: v.to(DEVICE) for k, v in batch.items()}
            test_logits.extend(model(**b).logits.float().cpu().numpy())
            if USE_TPU: xm.mark_step()

    del model, optimizer, scheduler, swa_states, ema_state, avg_state
    _empty_cache()
    return np.array(test_logits)

t_full = time.time()
full_test_logits = train_full_data()
print(f'Full-data done in {(time.time()-t_full)/60:.1f}min | Total: {(time.time()-t_start)/60:.1f}min')


In [ ]:
# ============================================================
# TEMPERATURE CALIBRATION  — [0.3, 3.0] (same as solution.ipynb)
# ============================================================
logits_t = torch.tensor(oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all, dtype=torch.long)

best_temp, best_nll = 1.0, float('inf')
for temp in np.arange(0.3, 3.0, 0.05):
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll: best_nll = nll; best_temp = temp

print(f'Temperature: {best_temp:.2f}  (searched [0.3, 3.0])')

# Blend fold-avg (ALPHA) + full-data (1-ALPHA) for test
oof_probs       = torch.softmax(logits_t / best_temp, dim=-1).numpy()
fold_test_probs = torch.softmax(
    torch.tensor(test_logits_fold,  dtype=torch.float32) / best_temp, dim=-1).numpy()
full_test_probs = torch.softmax(
    torch.tensor(full_test_logits, dtype=torch.float32) / best_temp, dim=-1).numpy()
test_probs      = ALPHA * fold_test_probs + (1 - ALPHA) * full_test_probs

oof_f1 = f1_score(y_all, oof_probs.argmax(1), average='macro')
print(f'OOF F1 (ModernBERT+RDrop+EMA): {oof_f1:.4f}')
print(classification_report(y_all, oof_probs.argmax(1),
      target_names=[LABEL_MAP[i] for i in range(NUM_LABELS)]))


In [ ]:
# ============================================================
# TF-IDF + CALIBRATED SVC  (5-fold)
# ============================================================
print('Building TF-IDF + SVC...')
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
word_tfidf = TfidfVectorizer(analyzer='word',    ngram_range=(1, 2),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
X_tr = sparse_hstack([char_tfidf.fit_transform(texts_train), word_tfidf.fit_transform(texts_train)])
X_te = sparse_hstack([char_tfidf.transform(texts_test),      word_tfidf.transform(texts_test)])

svc_oof  = np.zeros((len(y_all), NUM_LABELS))
svc_test = np.zeros((len(texts_test), NUM_LABELS))
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf5.split(np.zeros(len(y_all)), y_all)):
    clf = CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000), cv=3, method='sigmoid')
    clf.fit(X_tr[tr], y_all[tr])
    svc_oof[va] = clf.predict_proba(X_tr[va])
    svc_test   += clf.predict_proba(X_te) / 5
    print(f'  SVC fold {fi+1}: {f1_score(y_all[va], clf.predict(X_tr[va]), average="macro"):.4f}')

svc_f1 = f1_score(y_all, svc_oof.argmax(1), average='macro')
print(f'SVC OOF F1: {svc_f1:.4f}')


In [ ]:
# ============================================================
# ENSEMBLE — Global + Per-class Nelder-Mead (from solution.ipynb)
# Per-class weights let SVC help on Grok/Gemini while giving DeepSeek
# 100% ModernBERT (SVC F1~0.51 on DeepSeek poisons global blend).
# ============================================================
from scipy.optimize import minimize

print('Ensemble weight optimization...')
full_test_probs_ens = torch.softmax(
    torch.tensor(full_test_logits, dtype=torch.float32) / best_temp, dim=-1).numpy()
print(f'  Temperature applied: {best_temp:.2f}')

# ── Strategy 1: Global weights ───────────────────────────────────────────
def neg_macro_f1(weights, probs_list, labels):
    w = np.abs(weights); w = w / w.sum()
    blended = sum(wi * pi for wi, pi in zip(w, probs_list))
    return -f1_score(labels, blended.argmax(1), average='macro')

oof_components = [oof_probs, svc_oof]
best_result = None; best_neg_f1 = 0
for init in [[0.85,0.15],[0.80,0.20],[0.90,0.10],[0.75,0.25],[0.70,0.30],[0.95,0.05]]:
    result = minimize(neg_macro_f1, init, args=(oof_components, y_all),
                      method='Nelder-Mead',
                      options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-6})
    if best_result is None or result.fun < best_neg_f1:
        best_neg_f1 = result.fun; best_result = result

opt_w = np.abs(best_result.x) / np.abs(best_result.x).sum()
oof_blend_global = opt_w[0] * oof_probs + opt_w[1] * svc_oof
global_f1 = f1_score(y_all, oof_blend_global.argmax(1), average='macro')
print(f'  Global weights: MB={opt_w[0]:.3f}, SVC={opt_w[1]:.3f}  → OOF F1={global_f1:.4f}')

# ── Strategy 2: Per-class SVC weights ───────────────────────────────────
def neg_macro_f1_perclass(w_svc, mb_probs, svc_probs, labels):
    w = np.clip(w_svc, 0.0, 0.5)
    blended = np.zeros_like(mb_probs)
    for c in range(NUM_LABELS):
        blended[:, c] = (1 - w[c]) * mb_probs[:, c] + w[c] * svc_probs[:, c]
    return -f1_score(labels, blended.argmax(1), average='macro')

np.random.seed(SEED)
best_result_pc = None
for _ in range(12):
    init = np.random.uniform(0.05, 0.45, size=NUM_LABELS)
    result = minimize(neg_macro_f1_perclass, init,
                      args=(oof_probs, svc_oof, y_all),
                      method='Nelder-Mead',
                      options={'maxiter': 10000, 'xatol': 1e-5, 'fatol': 1e-7})
    if best_result_pc is None or result.fun < best_result_pc.fun:
        best_result_pc = result

opt_w_pc = np.clip(best_result_pc.x, 0.0, 0.5)
oof_blend_pc = np.zeros_like(oof_probs)
for c in range(NUM_LABELS):
    oof_blend_pc[:, c] = (1 - opt_w_pc[c]) * oof_probs[:, c] + opt_w_pc[c] * svc_oof[:, c]
perclass_f1 = f1_score(y_all, oof_blend_pc.argmax(1), average='macro')
print(f'  Per-class SVC weights: {dict(zip(LABEL_MAP.values(), opt_w_pc.round(3)))}')
print(f'  Per-class blend OOF F1: {perclass_f1:.4f}')

# ── Pick the better strategy ─────────────────────────────────────────────
use_perclass = perclass_f1 >= global_f1
if use_perclass:
    print(f'  >> Using PER-CLASS blend (+{perclass_f1 - global_f1:.4f} over global)')
    oof_blend = oof_blend_pc; oof_blend_f1 = perclass_f1
    best_alpha = 1 - opt_w_pc.mean()  # for reporting
else:
    print(f'  >> Using GLOBAL blend (per-class was {perclass_f1 - global_f1:+.4f})')
    oof_blend = oof_blend_global; oof_blend_f1 = global_f1
    best_alpha = opt_w[0]

# Build test blend
mbert_test = ALPHA * test_probs + (1 - ALPHA) * full_test_probs_ens
if use_perclass:
    test_blend = np.zeros_like(mbert_test)
    for c in range(NUM_LABELS):
        test_blend[:, c] = (1-opt_w_pc[c]) * mbert_test[:, c] + opt_w_pc[c] * svc_test[:, c]
else:
    test_blend = opt_w[0] * mbert_test + opt_w[1] * svc_test

print(f'\n  ModernBERT OOF: {oof_f1:.4f}  |  SVC OOF: {svc_f1:.4f}  |  Blend OOF: {oof_blend_f1:.4f}')


In [ ]:
# ============================================================
# THRESHOLD SWEEP  [0.60, 1.50] — wider range like solution.ipynb
# ============================================================
def threshold_sweep(probs, labels, lo=0.60, hi=1.50, steps=60, passes=8):
    best_mult = np.ones(NUM_LABELS)
    best_f1   = f1_score(labels, probs.argmax(1), average='macro')
    for _ in range(passes):
        improved = False
        for cls in range(NUM_LABELS):
            for m in np.linspace(lo, hi, steps):
                trial = best_mult.copy(); trial[cls] = m
                f1 = f1_score(labels, (probs * trial).argmax(1), average='macro')
                if f1 > best_f1 + 1e-6:
                    best_f1 = f1; best_mult = trial.copy(); improved = True
        if not improved: break
    return best_mult, best_f1

thresholds, thresh_f1 = threshold_sweep(oof_blend, y_all)
print(f'OOF F1 after threshold: {thresh_f1:.4f}  (was {oof_blend_f1:.4f})')
print('Multipliers:', {LABEL_MAP[i]: round(thresholds[i], 3) for i in range(NUM_LABELS)})

final_preds = (test_blend * thresholds).argmax(1)


In [ ]:
# ============================================================
# SUBMISSION  — integer labels (0-5), columns: ID + LABEL
# ============================================================
submission = pd.DataFrame({
    'ID':    range(len(final_preds)),
    'LABEL': final_preds.astype(int),
})
submission.to_csv('submission_v9.csv', index=False)

# Save probabilities for cross-notebook ensemble
np.save('test_blend_v9_tpu.npy', test_blend)    # (600, 6)
np.save('oof_blend_v9_tpu.npy',  oof_blend)     # (2400, 6)

print('submission_v9.csv:')
print(submission.head(10).to_string(index=False))
print(f'\nShape: {submission.shape}')
print('\nPrediction distribution:')
for i, nm in LABEL_MAP.items():
    cnt = (final_preds == i).sum()
    print(f'  {i} ({nm:10s}): {cnt:3d}  ({cnt/len(final_preds)*100:.1f}%)')

blend_type = 'per-class' if use_perclass else 'global'
print(f'\n{"="*55}')
print(f'FINAL SUMMARY  v9-TPU')
print(f'  ModernBERT OOF F1:   {oof_f1:.4f}')
print(f'  SVC OOF F1:          {svc_f1:.4f}')
print(f'  Blended OOF F1:      {oof_blend_f1:.4f}  ({blend_type})')
print(f'  + threshold nudge:   {thresh_f1:.4f}')
print(f'  Temperature:         {best_temp:.2f}')
print(f'  Fold scores: {[f"{s:.4f}" for s in fold_scores]}')
print(f'  Mean: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'{"="*55}')


In [ ]:
# ============================================================
# CROSS-NOTEBOOK ENSEMBLE  — TPU v9 + GPU solution.ipynb
# Run this after BOTH notebooks finish to get +0.002–0.005 F1
# ============================================================
# Step 1: save THIS notebook's test probabilities
np.save('test_blend_v9_tpu.npy',  test_blend)   # (600, 6) float32
np.save('oof_blend_v9_tpu.npy',   oof_blend)    # (2400, 6) for OOF comparison
print(f'Saved test_blend_v9_tpu.npy  shape={test_blend.shape}')
print(f'Saved oof_blend_v9_tpu.npy   shape={oof_blend.shape}')

# Step 2: in solution.ipynb, add this line right before the submission cell:
#   np.save('test_blend_v9_gpu.npy', test_blend)  # after threshold_sweep

# Step 3: once both .npy files are in /kaggle/working/, run the block below
# ─────────────────────────────────────────────────────────────────────────
import os as _os
_gpu_path = 'test_blend_v9_gpu.npy'
if _os.path.exists(_gpu_path):
    gpu_blend = np.load(_gpu_path)                # GPU notebook predictions
    tpu_blend = test_blend                         # this notebook's predictions

    # Grid search for best TPU/GPU mix on OOF isn't directly available
    # across notebooks, so use 50/50 as the starting point (models are
    # trained on identical data with different augmentations → decorrelated)
    best_combo_f1, best_w = 0.0, 0.5
    gpu_oof_path = 'oof_blend_v9_gpu.npy'
    if _os.path.exists(gpu_oof_path):
        gpu_oof = np.load(gpu_oof_path)
        for w in np.arange(0.30, 0.71, 0.05):
            combo_oof = w * oof_blend + (1 - w) * gpu_oof
            f1 = f1_score(y_all, combo_oof.argmax(1), average='macro')
            if f1 > best_combo_f1: best_combo_f1 = f1; best_w = w
        print(f'Best OOF blend: TPU={best_w:.2f}, GPU={1-best_w:.2f} → F1={best_combo_f1:.4f}')
    else:
        best_w = 0.50
        print(f'oof_blend_v9_gpu.npy not found — using 50/50 blend')

    combo_test = best_w * tpu_blend + (1 - best_w) * gpu_blend
    # Re-apply threshold sweep on combined predictions
    combo_preds = (combo_test * thresholds).argmax(1)
    sub_combo = pd.DataFrame({'ID': range(len(combo_preds)), 'LABEL': combo_preds.astype(int)})
    sub_combo.to_csv('submission_v9_combo.csv', index=False)
    print(f'Saved submission_v9_combo.csv  (TPU={best_w:.0%} + GPU={1-best_w:.0%})')
    print('Prediction distribution:')
    for i, nm in LABEL_MAP.items():
        cnt = (combo_preds == i).sum()
        print(f'  {i} ({nm:10s}): {cnt:3d}  ({cnt/len(combo_preds)*100:.1f}%)')
else:
    print(f'{_gpu_path} not found yet — run solution.ipynb first, then re-run this cell.')
    print('test_blend_v9_tpu.npy already saved — ready for cross-notebook blend.')


In [ ]:
# ============================================================
# WANDB VISUALIZATION — v9 TPU Results
# wandb proto conflict fix: pin to 0.18.7 (stable on Python 3.12)
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'wandb==0.18.7'], check=True)

# Must import fresh after reinstall — restart kernel if this still fails
import importlib, sys
for mod in list(sys.modules.keys()):
    if 'wandb' in mod:
        del sys.modules[mod]

import wandb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

WANDB_API_KEY = "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"
wandb.login(key=WANDB_API_KEY, relogin=True)

run = wandb.init(
    project="malto-competition",
    name="v9-tpu-modernbert-rdrop-ema",
    config={
        "model": MODEL_NAME, "n_folds": N_FOLDS,
        "ema_decay": EMA_DECAY, "kl_weight": KL_WEIGHT,
        "drw_cap": DRW_CAP, "lr": LR,
        "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
        "temperature": best_temp, "blend_alpha": best_alpha,
        "hardware": "TPU v5e-8", "dtype": "float32",
    }
)

CLASS_NAMES = [LABEL_MAP[i] for i in range(NUM_LABELS)]
COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']

# ── 1. Fold F1 bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(1, N_FOLDS+1), fold_scores, color=COLORS[:N_FOLDS], alpha=0.85, width=0.5)
ax.axhline(np.mean(fold_scores), color='red', ls='--', lw=1.5,
           label=f'Mean={np.mean(fold_scores):.4f}')
ax.axhline(0.96423, color='gold', ls=':', lw=1.5, label='1st place (0.96423)')
for bar, s in zip(bars, fold_scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{s:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0.92, 0.985); ax.set_xlabel('Fold'); ax.set_ylabel('Val F1 (macro)')
ax.set_title('5-Fold CV F1 — v9 TPU (ModernBERT + R-Drop + EMA)')
ax.legend(fontsize=9); ax.set_xticks(range(1, N_FOLDS+1))
plt.tight_layout(); wandb.log({"fold_f1_scores": wandb.Image(fig)}); plt.close()

# ── 2. Per-class F1 bar chart ────────────────────────────────────────────────
from sklearn.metrics import f1_score as _f1
per_class_mb  = [_f1(y_all==i, oof_probs.argmax(1)==i) for i in range(NUM_LABELS)]
per_class_bld = [_f1(y_all==i, final_preds==i)         for i in range(NUM_LABELS)]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(NUM_LABELS); w = 0.35
ax.bar(x-w/2, per_class_mb,  w, label='ModernBERT', color='steelblue',  alpha=0.85)
ax.bar(x+w/2, per_class_bld, w, label='Final blend', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=15)
ax.set_ylabel('Per-class F1'); ax.set_ylim(0, 1.05)
for i, (a, b) in enumerate(zip(per_class_mb, per_class_bld)):
    ax.text(i-w/2, a+0.01, f'{a:.3f}', ha='center', va='bottom', fontsize=7)
    ax.text(i+w/2, b+0.01, f'{b:.3f}', ha='center', va='bottom', fontsize=7)
ax.set_title('Per-class F1 — v9 TPU'); ax.legend(fontsize=9)
plt.tight_layout(); wandb.log({"per_class_f1": wandb.Image(fig)}); plt.close()

# ── 3. OOF Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_all, final_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, vmin=0, vmax=1, annot_kws={'size': 10})
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
ax.set_title(f'OOF Confusion Matrix — v9 TPU (F1={thresh_f1:.4f})')
plt.tight_layout(); wandb.log({"confusion_matrix": wandb.Image(fig)}); plt.close()

# ── 4. Training curves ────────────────────────────────────────────────────────
fold_history = [
    [(1,1.0137,0.8157),(2,0.5081,0.9215),(3,0.4596,0.9121),(4,0.5444,0.9409),
     (5,0.6844,0.9274),(6,0.9141,0.9428),(7,0.9156,0.9392),(8,0.9045,0.9473),
     (9,0.9100,0.9556),(10,0.9073,0.9556)],
    [(1,1.0896,0.7968),(2,0.5008,0.9010),(3,0.4584,0.9269),(4,0.5435,0.9336),
     (5,0.6829,0.9609),(6,0.9387,0.9354),(7,0.9223,0.9499)],
    [(1,1.0163,0.8066),(2,0.5012,0.8534),(3,0.4615,0.9515),(4,0.5426,0.9475),
     (5,0.6795,0.9615),(6,0.9174,0.9615),(7,0.9090,0.9615)],
    [(1,0.9583,0.8233),(2,0.5069,0.9098),(3,0.4571,0.9450),(4,0.5416,0.9192),
     (5,0.6848,0.9538),(6,0.9307,0.9359),(7,0.8955,0.9623),(8,0.9053,0.9538),
     (9,0.8963,0.9538)],
    [(1,0.9892,0.7973),(2,0.5075,0.8095),(3,0.4616,0.9267),(4,0.5441,0.9496),
     (5,0.6807,0.9548),(6,0.9071,0.9712),(7,0.8969,0.9499),(8,0.8987,0.9573)],
]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for fi, hist in enumerate(fold_history):
    ep = [h[0] for h in hist]
    axes[0].plot(ep, [h[1] for h in hist], marker='o', ms=4,
                 color=COLORS[fi], alpha=0.85, label=f'Fold {fi+1}')
    axes[1].plot(ep, [h[2] for h in hist], marker='o', ms=4,
                 color=COLORS[fi], alpha=0.85, label=f'Fold {fi+1}')
for ax, title, yl in zip(axes, ['Training Loss', 'Validation F1'], ['Loss', 'Macro F1']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(yl)
    ax.set_title(f'{title} per Epoch — v9 TPU')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[1].axhline(np.mean(fold_scores), color='red', ls='--', lw=1,
                label=f'Mean={np.mean(fold_scores):.4f}')
plt.tight_layout(); wandb.log({"training_curves": wandb.Image(fig)}); plt.close()

# ── 5. Scalar metrics ────────────────────────────────────────────────────────
wandb.log({
    "oof_f1_modernbert": oof_f1,
    "oof_f1_svc":        svc_f1,
    "oof_f1_blend":      best_blend_f1,
    "oof_f1_final":      thresh_f1,
    "fold_mean":         float(np.mean(fold_scores)),
    "fold_std":          float(np.std(fold_scores)),
    **{f"fold_{i+1}_f1": s for i, s in enumerate(fold_scores)},
    **{f"per_class_f1/{CLASS_NAMES[i]}": per_class_bld[i] for i in range(NUM_LABELS)},
})

run.finish()
print('WandB run complete. Check your dashboard at https://wandb.ai')
